# **LAB 4: LLS and Prompt Engineering for Decision Support**

### Part 0: Repository and API-key Setup

In [56]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]


# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


## **Section 1 - Talking to an LLM Programmatically**

### Part 1.1 - Your first API call

In [57]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content, response.usage


# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?
answer, usage  = ask_llm("What is microfinance in one sentence?")
print(answer)
print(usage)

Microfinance refers to the provision of small-scale financial services, such as loans, savings, and insurance, to low-income individuals or groups who lack access to traditional banking services, with the goal of promoting financial inclusion and economic empowerment.
CompletionUsage(completion_tokens=47, prompt_tokens=49, total_tokens=96, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041768125, prompt_time=0.00218431, completion_time=0.147801238, total_time=0.149985548)


**Anatomy of a call**

1. What is the difference between the system and user roles? Give an example of something that belongs in each.


2. What is a token, roughly? Why do API providers bill per token rather than per request?

### Part 1.2 - Temperature: the randomness dial

In [58]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
question = "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.
print('===== Temperature: 0.0 ======')
for i in range(5):
    answer = ask_llm(question, temperature = 0.0)
    answer = answer[0]
    print(f"Run {i+1}: {answer}")

print()
print('===== Temperature: 1.2 =====')
for i in range(5):
    answer = ask_llm(question, temperature = 1.2)
    answer = answer[0]
    print(f"Run {i+1}: {answer}")

===== Temperature: 0.0 ======
Run 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **TradeUp Savings**: This name suggests that the savings product will help traders "trade up" and improve their financial situation.
3. **Sika Kokoo**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Kokoo" means "gather" or "collect", so this name could convey the idea of gathering savings.
4. **MarketMoen**: This name incorporates "moen", a Ghanaian Pidgin word for "money", and "market", to create a catchy and memorable name.
5. **Adanfo Savings**: "Adanfo" means "helpers" or "supporters" in the Akan language, which could suggest that the savings product is a helpful tool for market traders.
6. **Kae Dua**: "Kae" means "grow" or "increase" in the Akan language, and "dua" means "money" or "wealth". This name could convey the idea 

**Temperature** 

1. What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?



## **Section 2 - The Dataset: Loan Application Letters**

In [59]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


## **Section 3 - Prompt Engineering for the Decision Support System**

### Part 3.1 - Component 1: Summarization

In [60]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this:"

print("===== L002 V1 =====")
print(ask_llm(LETTERS["L002"], system_prompt=SUMMARY_PROMPT_V1))

print()
print("===== L006 V1 =====")
print(ask_llm(LETTERS["L006"], system_prompt=SUMMARY_PROMPT_V1))

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_PROMPT_V2 = """You are an assistant to a microfinance loan officer.
Summarize loan applications in 3-4 sentences. Be factual and neutral.
Do not invent any details not stated in the letter."""

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print()
print("===== L002 V2 =====")
print(ask_llm(f"Summarize this loan application:\n\n{LETTERS['L002']}", system_prompt=SUMMARY_PROMPT_V2, temperature=0))

print()
print("===== L006 V2 =====")
print(ask_llm(f"Summarize this loan application:\n\n{LETTERS['L006']}", system_prompt=SUMMARY_PROMPT_V2, temperature=0))

===== L002 V1 =====
("Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in business, but expects it to improve after the festive season. He doesn't have collateral, but is hoping to repay the loan as soon as his finances improve. He's requesting urgent assistance.", CompletionUsage(completion_tokens=82, prompt_tokens=127, total_tokens=209, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041486444, prompt_time=0.006198245, completion_time=0.274945616, total_time=0.281143861))

===== L006 V1 =====
('Kofi, a 22-year-old, is requesting a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He has no prior experience in these ventures but claims to be "business-minded" based on his friends\' opinions. He promises to repay the loan within one year, once his businesses are succ

 **Summarization prompts**
 1. What concrete problems did V1's output have that V2 fixed? Quote examples. 
 
 
 2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

### Part 3.2 - Component 2: Structured exctraction (JSON)

In [ ]:
import json
import pandas as pd

# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
EXTRACT_PROMPT = """You are a data extraction assistant for a microfinance institution.
Extract information from loan application letters and return Only a JSON object with the following keys:
    applicant_name (string), amount_ghs (number), purpose(string), monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean), repayment_months (number or null)
    
If a field is not stated in the letter, use null. Do not guess.

Example Letter: "My name is Ama Owusu. I run a hair salon in Tema and need GHS 5,000 to buy new equipment. I make about GHS 600 profit monthly. My husband will serve as guarantor. I will repay over 12 months."

Output: {
    "applicant_name": "Ama Owusu",
    "amount_ghs": 5000,
    "purpose": "buy new equipment for hair salon",
    "monthly_profit_ghs": 600,
    "has_collateral_or_guarantor": true,
    "repayment_months": 12
}

Return ONLY the JSON object, nothing else."""


# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
    response = ask_llm(
        f"Extract fields from this loan application:\n\n{letter_text}",
        system_prompt=EXTRACT_PROMPT,
        temperature=0
    )
    response = response[0]
    print("RAW RESPONSE:", response)
    try:
        clean = response.replace("```json", "").replace("```", "").strip()
        return json.loads(clean)
    except:
        print("Warning: could not parse response")
        return None


# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
results = {}
for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_text)
    if result:
        results[letter_id] = result

df = pd.DataFrame(results).T
print(df)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kzsq38eaeczrwd9q1x5xx7bk` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99841, Requested 494. Please try again in 4m49.44s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

**Structured Extraction**

1. Why must the few-shot example NOT come from the six letters you are processing? 


2. Why "use null, do not guess" — what did the model do without that instruction?



3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?

### Part 3.3 - Component 3: The decision-support brief

In [ ]:
#TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_PROMPT = """You are an assistant at a microfinance loan office.
Your job is to help the officer analyze loan applications, not to make final decisions.
Final approval or rejection decisions are made by humans only.

Given a loan application and extracted data, produce a structured brief with these sections:
    1. Strengths (bullet points, grounded in the letter)
    2. Risks / red flags (bullet points)
    3. Missing information that the officer should request
    4. Suggested next step (e.g. "invite for interview", "request documents", "flag for senior review") - do not say approve or reject."""


# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
def generate_brief(letter_text, extracted_json):
    user_prompt = f"""Loan Application Letter:
    {letter_text}

Extraxted Data:
{json.dumps(extracted_json, indent = 3)}

Produce the brief."""

    response = ask_llm(user_prompt, system_prompt= BRIEF_PROMPT, temperature = 0.3)
    return response[0]

briefs = {}
for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)
    briefs[letter_id] = generate_brief(letter_text, extracted)

for letter_id in ["L001", "L002", "L006"]:
    print(f"\n===== {letter_id} =====")
    print(briefs[letter_id])



RAW RESPONSE: {
    "application_name": "Akosua Mensah",
    "amount_ghs": 8000,
    "purpose": "buy a deep freezer and expand into frozen foods",
    "monthly_profit_ghs": 900,
    "has_collateral_or_guarantor": true,
    "repayment_months": 20
}
RAW RESPONSE: {
    "application_name": "Kwame Boateng",
    "amount_ghs": 25000,
    "purpose": "repair trotro engine and settle some personal debts",
    "monthly_profit_ghs": null,
    "has_collateral_or_guarantor": false,
    "repayment_months": null
}
RAW RESPONSE: {
    "application_name": "Efua Darko",
    "amount_ghs": 15000,
    "purpose": "purchase two industrial sewing machines and fabric stock",
    "monthly_profit_ghs": 2800,
    "has_collateral_or_guarantor": true,
    "repayment_months": 15
}
RAW RESPONSE: {
    "application_name": "Yaw Owusu",
    "amount_ghs": 12000,
    "purpose": "for feed and 500 new layers for poultry farm",
    "monthly_profit_ghs": 1500,
    "has_collateral_or_guarantor": true,
    "repayment_months": 1

**Decision support** 

1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the system identify the right strengths and red flags in each?

2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and one ethical reason.

## **Section 4 - Evaluation: Quality, Reliability, Appropriateness**

### Part 4.1 - Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs", 
          "has_collateral_or_guarantor", "repayment_months"]

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
     

### Part 4.2 - Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 - Hallucinating probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.


**Evaluation results**

1. Report your extraction accuracy. Which field was hardest for the model and why? 

2. What did the reliability experiment show about temperature and production systems? 

3. Did your system hallucinate under probing? If yes, how could the prompt (or the system design around it) reduce the risk?

### Part 4.4 — Appropriateness: should this system exist?